# 02: 時系列予測モデルの構築とデプロイ

このノートブックでは、小売EC需要予測に用いる時系列モデルを DataRobot AutoTS で構築し、デプロイします。

## 処理の流れ
1. 環境変数の読み込みと DataRobot クライアントの初期化
2. Use Case の作成または取得
3. 学習データのアップロード (AI Catalog)
4. AutoTS プロジェクトの作成・学習
5. 推奨モデルの取得と精度確認
6. モデルの登録とデプロイ
7. スコアリング用データセットの作成とアップロード
8. `.env` に設定する ID の出力

## 1. 環境変数の読み込み

In [ ]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv("../.env", override=True)

DATAROBOT_API_TOKEN = os.environ["DATAROBOT_API_TOKEN"]
DATAROBOT_ENDPOINT = os.environ.get("DATAROBOT_ENDPOINT", "https://app.datarobot.com/api/v2")

print(f"Endpoint: {DATAROBOT_ENDPOINT}")
print(f"Token:    {DATAROBOT_API_TOKEN[:8]}...")

## 2. DataRobot クライアントの初期化

In [ ]:
import datarobot as dr

dr.Client(
    token=DATAROBOT_API_TOKEN,
    endpoint=DATAROBOT_ENDPOINT,
)
print(f"DataRobot client initialized (v{dr.__version__})")

## 3. Use Case の作成または取得

`.env` に `DATAROBOT_DEFAULT_USE_CASE` が設定されていればそれを使い、なければ新規作成します。

In [ ]:
USE_CASE_NAME = "小売EC需要予測エージェント"

existing_use_case_id = os.environ.get("DATAROBOT_DEFAULT_USE_CASE", "").strip()

if existing_use_case_id:
    use_case = dr.UseCase.get(existing_use_case_id)
    print(f"既存の Use Case を取得しました: {use_case.name} ({use_case.id})")
else:
    use_case = dr.UseCase.create(name=USE_CASE_NAME)
    print(f"新規 Use Case を作成しました: {use_case.name} ({use_case.id})")

print(f"\nUse Case ID: {use_case.id}")

## 4. 学習データを AI Catalog にアップロード

`../data/retail_sales_dataset.csv` を AI Catalog にアップロードします。既にアップロード済みの場合はスキップします。

In [ ]:
import pathlib

DATASET_PATH = pathlib.Path("../data/retail_sales_dataset.csv").resolve()
DATASET_NAME = "retail_sales_dataset"

assert DATASET_PATH.exists(), f"データファイルが見つかりません: {DATASET_PATH}"
print(f"データファイル: {DATASET_PATH}")

In [ ]:
# 既にカタログに存在するか確認
catalog_datasets = dr.Dataset.list()
existing = [d for d in catalog_datasets if d.name == DATASET_NAME]

if existing:
    dataset = existing[0]
    print(f"既存のデータセットを使用します: {dataset.name} (ID: {dataset.id})")
else:
    dataset = dr.Dataset.create_from_file(file_path=str(DATASET_PATH))
    print(f"データセットをアップロードしました: {dataset.name} (ID: {dataset.id})")

# Use Case に紐付け
try:
    use_case.add(dataset)
    print("Use Case にデータセットを紐付けました。")
except Exception as e:
    print(f"紐付け済みまたはスキップ: {e}")

## 5. AutoTS プロジェクトの作成

以下の設定で時系列自動モデリングを開始します:

| パラメータ | 値 |
|---|---|
| ターゲット | `sales_billion_yen` |
| 日付カラム | `year_month` |
| シリーズID | `store_type` |
| 特徴量導出ウィンドウ | 12ヶ月 |
| 予測ウィンドウ | 3ヶ月 |
| バックテスト数 | 3 |

In [ ]:
PROJECT_NAME = "小売EC需要予測_時系列モデル"

# プロジェクト作成
project = dr.Project.create_from_dataset(
    dataset_id=dataset.id,
    project_name=PROJECT_NAME,
    use_case=use_case,
)
print(f"プロジェクトを作成しました: {project.project_name} (ID: {project.id})")

In [ ]:
# 時系列パーティション設定
from datarobot.helpers.partitioning_methods import DatetimePartitioningSpecification

ts_spec = DatetimePartitioningSpecification(
    datetime_partition_column="year_month",
    multiseries_id_columns=["store_type"],
    feature_derivation_window_start=-12,
    feature_derivation_window_end=0,
    forecast_window_start=1,
    forecast_window_end=3,
    number_of_backtests=3,
    use_time_series=True,
)

print("時系列パーティション設定:")
print(f"  特徴量導出ウィンドウ: -12 ~ 0 ヶ月")
print(f"  予測ウィンドウ: 1 ~ 3 ヶ月")
print(f"  バックテスト数: 3")

In [ ]:
# Autopilot 開始
project.analyze_and_model(
    target="sales_billion_yen",
    partitioning_method=ts_spec,
    mode=dr.enums.AUTOPILOT_MODE.FULL_AUTO,
    worker_count=-1,
)
print("Autopilot を開始しました。完了まで待機します...")

## 6. Autopilot 完了を待機

モデル構築の完了を待ちます。時系列モデルの場合、通常10-30分程度かかります。

In [ ]:
project.wait_for_autopilot(check_interval=30, verbosity=1)
print("\nAutopilot が完了しました。")

## 7. 推奨モデルの取得と精度確認

In [ ]:
# 推奨モデルを取得
recommendation = dr.ModelRecommendation.get(project.id)
best_model = dr.Model.get(project=project.id, model_id=recommendation.model_id)

print(f"推奨モデル: {best_model.model_type}")
print(f"モデル ID:  {best_model.id}")
print(f"\n精度指標:")
if best_model.metrics:
    for metric_name, metric_data in best_model.metrics.items():
        if isinstance(metric_data, dict) and metric_data.get("backtesting") is not None:
            print(f"  {metric_name}: {metric_data['backtesting']:.6f} (backtesting)")
        elif isinstance(metric_data, dict) and metric_data.get("validation") is not None:
            print(f"  {metric_name}: {metric_data['validation']:.6f} (validation)")

## 8. モデルの登録 (Model Registry)

In [ ]:
from datarobot.models.model_registry import RegisteredModel

registered = RegisteredModel.create_for_leaderboard_item(
    model_id=best_model.id,
    name="小売EC需要予測_時系列モデル",
    registered_model_name="小売EC需要予測_時系列モデル",
)

# RegisteredModel から version を取得
versions = registered.list_versions()
latest_version = versions[0]

print(f"モデルを登録しました。")
print(f"  Registered Model ID: {registered.id}")
print(f"  Version ID: {latest_version.id}")

## 9. モデルのデプロイ

登録したモデルを DataRobot にデプロイします。

In [ ]:
prediction_environment = dr.PredictionEnvironment.list()[0]
print(f"予測環境: {prediction_environment.name} (ID: {prediction_environment.id})")

In [ ]:
deployment = dr.Deployment.create(
    model_package_id=latest_version.id,
    label="小売EC需要予測_時系列デプロイメント",
    default_prediction_server_id=prediction_environment.id,
    use_case_id=use_case.id,
)

FORECAST_DEPLOYMENT_ID = deployment.id
print(f"デプロイが完了しました。")
print(f"  Deployment ID: {FORECAST_DEPLOYMENT_ID}")

## 10. スコアリング用データセットの作成

時系列予測には、直近の実績データに加えて未来の日付行が必要です。ここでは、学習データの末尾に 3ヶ月分の未来行を追加したスコアリング用データセットを作成します。

In [ ]:
import pandas as pd

# 元データを読み込み
df = pd.read_csv(str(DATASET_PATH))
df["year_month"] = pd.to_datetime(df["year_month"])

print(f"元データ: {len(df)} 行")
print(f"日付範囲: {df['year_month'].min()} ~ {df['year_month'].max()}")
print(f"シリーズ: {df['store_type'].unique().tolist()}")

# 各シリーズの直近12ヶ月を取得 (特徴量導出ウィンドウ分)
max_date = df["year_month"].max()
cutoff_date = max_date - pd.DateOffset(months=11)
recent_df = df[df["year_month"] >= cutoff_date].copy()

print(f"\n直近12ヶ月のデータ: {len(recent_df)} 行")
print(f"日付範囲: {recent_df['year_month'].min()} ~ {recent_df['year_month'].max()}")

# 未来3ヶ月の行を追加
future_rows = []
for series in df["store_type"].unique():
    for i in range(1, 4):
        future_date = max_date + pd.DateOffset(months=i)
        row = {"year_month": future_date, "store_type": series}
        # 他のカラムは NaN (予測対象)
        for col in df.columns:
            if col not in ["year_month", "store_type"]:
                row[col] = None
        future_rows.append(row)

future_df = pd.DataFrame(future_rows)
scoring_df = pd.concat([recent_df, future_df], ignore_index=True)
scoring_df = scoring_df.sort_values(["store_type", "year_month"]).reset_index(drop=True)

print(f"\nスコアリングデータ: {len(scoring_df)} 行")
print(f"日付範囲: {scoring_df['year_month'].min()} ~ {scoring_df['year_month'].max()}")

# CSV保存
SCORING_PATH = pathlib.Path("../data/retail_sales_scoring.csv").resolve()
scoring_df.to_csv(str(SCORING_PATH), index=False)
print(f"\n保存先: {SCORING_PATH}")

## 11. スコアリングデータを AI Catalog にアップロード

In [ ]:
SCORING_DATASET_NAME = "retail_sales_scoring"

# 既存チェック
catalog_datasets = dr.Dataset.list()
existing_scoring = [d for d in catalog_datasets if d.name == SCORING_DATASET_NAME]

if existing_scoring:
    scoring_dataset = existing_scoring[0]
    print(f"既存のスコアリングデータセットを使用します: {scoring_dataset.name} (ID: {scoring_dataset.id})")
else:
    scoring_dataset = dr.Dataset.create_from_file(file_path=str(SCORING_PATH))
    print(f"スコアリングデータセットをアップロードしました: {scoring_dataset.name} (ID: {scoring_dataset.id})")

# Use Case に紐付け
try:
    use_case.add(scoring_dataset)
    print("Use Case にスコアリングデータセットを紐付けました。")
except Exception as e:
    print(f"紐付け済みまたはスキップ: {e}")

SCORING_DATASET_ID = scoring_dataset.id
print(f"\nScoring Dataset ID: {SCORING_DATASET_ID}")

## 12. `.env` に設定する変数の出力

以下の値を `.env` ファイルに追記してください。

In [ ]:
print("=" * 60)
print("以下を .env ファイルに追記してください:")
print("=" * 60)
print(f"FORECAST_DEPLOYMENT_ID={FORECAST_DEPLOYMENT_ID}")
print(f"SCORING_DATASET_ID={SCORING_DATASET_ID}")
print("=" * 60)
print(f"\nプロジェクト ID: {project.id}")
print(f"Use Case ID:    {use_case.id}")
print("\n完了: 次のステップは 03_create_vdb.ipynb に進んでください。")